In [2]:
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    random_split,
)

from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from astropy.stats import mad_std

from sklearn.metrics import precision_recall_curve

import configparser

In [3]:
class LAEDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial = nn.Sequential(
            nn.Conv3d(
                1,
                16,
                kernel_size=(1, 3, 3),
                padding=(0, 1, 1),
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
            nn.Conv3d(
                16,
                32,
                kernel_size=(1, 3, 3),
                padding=(0, 1, 1),
            ),
            nn.InstanceNorm3d(
                32,
                affine=True,
            ),
            nn.GELU(),
            nn.AdaptiveMaxPool3d((None, 1, 1)),
        )
        self.spectral = nn.Sequential(

            nn.Conv1d(
                32,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.GELU(),
            nn.Conv1d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.GELU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(DROPOUT),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        x = self.spatial(x)
        x = x.squeeze(-1).squeeze(-1)
        x = self.spectral(x)
        x = self.head(x)
        return x


class LAEDetector3D(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1),
            nn.InstanceNorm3d(16, affine=True),
            nn.GELU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.InstanceNorm3d(32, affine=True),
            nn.GELU(),
            
            nn.AdaptiveMaxPool3d((1, 1, 1))
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(DROPOUT),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x

### Train the CNN

In [1]:
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    random_split,
)

from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from astropy.stats import mad_std

from sklearn.metrics import precision_recall_curve

import configparser

# ============================================================================
# CONFIG
# ============================================================================

config = configparser.ConfigParser()
config.read("config.ini")

warnings.filterwarnings("ignore", category=UserWarning)

CUBE_FILE   = "./cube.fits"
TRUE_CAT    = "./regions_tabelle.fits"
FALSE_CAT   = "./false_examples.fits"

MODEL_FILE  = "lae_model.pt"
THRESH_FILE = "lae_threshold.txt"


LYA_REST = 1215.67
SEED     = 42

HALF_Z = config.getint("TRAINING", "HALF_Z")
HALF_Y = config.getint("TRAINING", "HALF_Y")
HALF_X = config.getint("TRAINING", "HALF_X")

BATCH_SIZE = 128
EPOCHS     = 50

# WICHTIG:
LR = 1e-4

DROPOUT     = config.getfloat("CNN", "DROPOUT")
VAL_SPLIT   = 0.15
TEST_SPLIT  = 0.10
PATIENCE    = 10
WEIGHT_DECAY = 1e-4

POS_WEIGHT = config.getfloat("TRAINING", "POS_WEIGHT")

N_HARD_NEGATIVES = config.getint("TRAINING", "N_HARD_NEG")

TARGET_PRECISION = config.getfloat("TRAINING", "PRECISION")

def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class AugmentDataset(torch.utils.data.Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):

        x, y = self.dataset[idx]

        # flip
        if np.random.rand() > 0.5:
            x = torch.flip(x, dims=[2])

        if np.random.rand() > 0.5:
            x = torch.flip(x, dims=[3])

        # rotation
        k = np.random.randint(0, 4)
        x = torch.rot90(x, k=k, dims=[2, 3])

        # multiplicative scaling
        if np.random.rand() > 0.5:
            scale = np.random.uniform(0.9, 1.1)
            x = x * scale

        # additive noise
        if np.random.rand() > 0.5:
            x = x + torch.randn_like(x) * 0.03

        return x, y

    def __len__(self):
        return len(self.dataset)

class FocalLoss(nn.Module):

    def __init__(self, gamma=2.0, pos_weight=1.0):

        super().__init__()

        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):

        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        probs = torch.sigmoid(logits)

        pt = torch.where(
            targets == 1,
            probs,
            1 - probs,
        )

        alpha = torch.where(
            targets == 1,
            torch.full_like(targets, self.pos_weight),
            torch.ones_like(targets),
        )

        loss = alpha * (1 - pt) ** self.gamma * bce

        return loss.mean()


class LAEDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial = nn.Sequential(
            nn.Conv3d(
                1,
                16,
                kernel_size=(1, 3, 3),
                padding=(0, 1, 1),
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
            nn.Conv3d(
                16,
                32,
                kernel_size=(1, 3, 3),
                padding=(0, 1, 1),
            ),
            nn.InstanceNorm3d(
                32,
                affine=True,
            ),
            nn.GELU(),
            nn.AdaptiveMaxPool3d((None, 1, 1)),
        )
        self.spectral = nn.Sequential(

            nn.Conv1d(
                32,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.GELU(),
            nn.Conv1d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.GELU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(DROPOUT),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        x = self.spatial(x)
        x = x.squeeze(-1).squeeze(-1)
        x = self.spectral(x)
        x = self.head(x)
        return x


class LAEDetector3D(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1),
            nn.InstanceNorm3d(16, affine=True),
            nn.GELU(),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.InstanceNorm3d(32, affine=True),
            nn.GELU(),
            
            nn.AdaptiveMaxPool3d((1, 1, 1))
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(DROPOUT),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x

def get_wcs_axis_order(wcs):

    axis_names = [n.upper() for n in wcs.axis_type_names]

    print(f"  WCS-Achsen erkannt: {axis_names}")

    ra_ax = next(
        (i for i, n in enumerate(axis_names) if "RA" in n),
        None,
    )

    dec_ax = next(
        (i for i, n in enumerate(axis_names) if "DEC" in n),
        None,
    )

    wave_ax = next(
        (
            i
            for i, n in enumerate(axis_names)
            if any(k in n for k in ("WAVE", "LAMBDA"))
        ),
        None,
    )

    if None in (ra_ax, dec_ax, wave_ax):
        raise RuntimeError("WCS-Achsen nicht erkannt")

    return ra_ax, dec_ax, wave_ax

def extract_subcubes(
    cube_data,
    wcs,
    catalog_path,
    label,
):

    cat = Table.read(catalog_path)

    ra_ax, dec_ax, wave_ax = get_wcs_axis_order(wcs)

    max_z, max_y, max_x = cube_data.shape

    dz = HALF_Z
    dy = HALF_Y
    dx = HALF_X

    subs = []
    labels = []

    skipped = 0

    for row in cat:

        ra  = float(row["RA"])
        dec = float(row["DEC"])
        z   = float(row["REDSHIFT"])

        wave_obs = LYA_REST * (1 + z)

        world = [None, None, None]

        world[ra_ax]   = ra
        world[dec_ax]  = dec
        world[wave_ax] = wave_obs

        try:
            pixel = wcs.all_world2pix([world], 0)[0]

        except Exception:
            skipped += 1
            continue

        px = int(round(pixel[ra_ax]))
        py = int(round(pixel[dec_ax]))
        pz = int(round(pixel[wave_ax]))

        if not (
            dz <= pz < max_z - dz and
            dy <= py < max_y - dy and
            dx <= px < max_x - dx
        ):
            skipped += 1
            continue

        sub = cube_data[
            pz-dz:pz+dz+1,
            py-dy:py+dy+1,
            px-dx:px+dx+1,
        ].astype(np.float32)

        sub = np.nan_to_num(sub, nan=0.0)

        # WICHTIG:
        # KEIN median-subtraction mehr

        local_std = mad_std(sub) + 1e-6

        sub = sub / local_std

        subs.append(sub)
        labels.append(label)

    if skipped:
        print(f"  -> {skipped} Quellen übersprungen")

    X = np.expand_dims(
        np.array(subs, dtype=np.float32),
        axis=1,
    )

    y = np.array(labels, dtype=np.float32)

    return X, y

def sample_hard_negatives(
    cube_data,
    n_samples,
    seed=SEED,
):

    rng = np.random.default_rng(seed)

    max_z, max_y, max_x = cube_data.shape

    dz = HALF_Z
    dy = HALF_Y
    dx = HALF_X

    subs = []

    while len(subs) < n_samples:

        z = rng.integers(dz, max_z - dz)
        y = rng.integers(dy, max_y - dy)
        x = rng.integers(dx, max_x - dx)

        sub = cube_data[
            z-dz:z+dz+1,
            y-dy:y+dy+1,
            x-dx:x+dx+1,
        ].astype(np.float32)

        sub = np.nan_to_num(sub, nan=0.0)

        local_std = mad_std(sub) + 1e-6

        if local_std < 1e-6:
            continue

        sub = sub / local_std

        subs.append(sub)

    X = np.expand_dims(
        np.array(subs, dtype=np.float32),
        axis=1,
    )

    y = np.zeros(n_samples, dtype=np.float32)

    return X, y

def train_model(
    model,
    dl_train,
    dl_val,
    device,
):

    criterion = FocalLoss(
        gamma=2.0,
        pos_weight=POS_WEIGHT,
    ).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
    )

    best_loss = np.inf
    best_state = None

    patience_counter = 0

    print(f"\n{'Epoche':>6} | {'Train':>10} | {'Val':>10}")
    print("-" * 36)

    for epoch in range(1, EPOCHS + 1):

        model.train()

        train_loss = 0.0

        for bx, by in dl_train:

            bx = bx.to(device)
            by = by.to(device).unsqueeze(1)

            optimizer.zero_grad()

            logits = model(bx)

            loss = criterion(logits, by)

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(dl_train)

        model.eval()

        val_loss = 0.0

        all_probs = []

        with torch.no_grad():

            for bx, by in dl_val:

                bx = bx.to(device)
                by = by.to(device).unsqueeze(1)

                logits = model(bx)

                loss = criterion(logits, by)

                val_loss += loss.item()

                probs = torch.sigmoid(logits)

                all_probs.extend(
                    probs.cpu().numpy().flatten()
                )

        val_loss /= len(dl_val)

        scheduler.step()

        prob_std = np.std(all_probs)

        print(
            f"{epoch:>6} | "
            f"{train_loss:>10.4f} | "
            f"{val_loss:>10.4f} | "
            f"prob_std={prob_std:.4f}"
        )

        if val_loss < best_loss:

            best_loss = val_loss

            best_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

            if patience_counter >= PATIENCE:

                print("\nEarly stopping")

                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

def evaluate_model(
    model,
    dl,
    device,
    threshold=None,
):

    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for bx, by in dl:

            probs = torch.sigmoid(
                model(bx.to(device))
            ).cpu().numpy().flatten()

            all_probs.extend(probs)

            all_labels.extend(
                by.numpy().astype(int)
            )

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    if threshold is None:

        prec, rec, thresholds = precision_recall_curve(
            all_labels,
            all_probs,
        )

        idx = np.where(
            prec[:-1] >= TARGET_PRECISION
        )[0]

        if len(idx) > 0:

            best_idx = idx[np.argmax(rec[idx])]

            threshold = float(thresholds[best_idx])

            print(f"\nThreshold: {threshold:.3f}")
            print(f"Precision: {prec[best_idx]:.3f}")
            print(f"Recall   : {rec[best_idx]:.3f}")

        else:

            threshold = 0.5

    preds = (all_probs >= threshold).astype(int)

    tp = ((preds == 1) & (all_labels == 1)).sum()
    fp = ((preds == 1) & (all_labels == 0)).sum()
    tn = ((preds == 0) & (all_labels == 0)).sum()
    fn = ((preds == 0) & (all_labels == 1)).sum()

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)

    f1 = (
        2 * precision * recall /
        (precision + recall + 1e-9)
    )

    print("\nEvaluation")
    print("-" * 30)

    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1        : {f1:.3f}")

    print("\nConfusion Matrix")
    print(f"TP={tp}  FP={fp}")
    print(f"FN={fn}  TN={tn}")

    return threshold

def main():

    set_seed()

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"Gerät: {device}")

    print("\n[1/5] Lade Cube")

    with fits.open(CUBE_FILE, memmap=False) as hdul:

        data_hdu = next(
            (
                h for h in hdul
                if h.data is not None and h.data.ndim == 3
            ),
            hdul[0],
        )

        cube_data = np.array(
            data_hdu.data,
            dtype=np.float32,
        )

        cube_data = np.nan_to_num(
            cube_data,
            nan=0.0,
        )

        wcs = WCS(data_hdu.header)

    print(f"Cube Shape: {cube_data.shape}")

    print("\n[2/5] Positive Samples")

    X_pos, y_pos = extract_subcubes(
        cube_data,
        wcs,
        TRUE_CAT,
        1.0,
    )

    print(f"Positive: {len(y_pos)}")

    print("\n[3/5] Hard Negatives")

    X_neg, y_neg = sample_hard_negatives(
        cube_data,
        N_HARD_NEGATIVES,
    )

    print(f"Negative: {len(y_neg)}")

    del cube_data

    X = np.concatenate([X_pos, X_neg])
    y = np.concatenate([y_pos, y_neg])

    dataset = TensorDataset(
        torch.tensor(X),
        torch.tensor(y),
    )

    val_size = int(len(dataset) * VAL_SPLIT)
    test_size = int(len(dataset) * TEST_SPLIT)

    train_size = len(dataset) - val_size - test_size

    ds_train, ds_val, ds_test = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    ds_train = AugmentDataset(ds_train)

    dl_train = DataLoader(
        ds_train,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    dl_val = DataLoader(
        ds_val,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    dl_test = DataLoader(
        ds_test,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    print(f"\nTrain: {len(ds_train)}")
    print(f"Val  : {len(ds_val)}")
    print(f"Test : {len(ds_test)}")

    print("\n[4/5] Trainiere Modell")

    model = LAEDetector3D().to(device)

    model = train_model(
        model,
        dl_train,
        dl_val,
        device,
    )

    torch.save(
        model.state_dict(),
        MODEL_FILE,
    )

    print(f"\nModell gespeichert: {MODEL_FILE}")

    print("\n[5/5] Evaluation")

    threshold = evaluate_model(
        model,
        dl_val,
        device,
    )

    with open(THRESH_FILE, "w") as f:
        f.write(str(threshold))

    print(f"\nThreshold gespeichert: {threshold:.3f}")

    print("\nTESTSET")

    evaluate_model(
        model,
        dl_test,
        device,
        threshold=threshold,
    )

    print("done")

if __name__ == "__main__":
    main()

Gerät: cpu

[1/5] Lade Cube
Cube Shape: (1036, 2478, 2260)

[2/5] Positive Samples
  WCS-Achsen erkannt: ['RA', 'DEC', 'WAVE']
  -> 1 Quellen übersprungen
Positive: 4251

[3/5] Hard Negatives
Negative: 1600

Train: 4389
Val  : 877
Test : 585

[4/5] Trainiere Modell

Epoche |      Train |        Val
------------------------------------
     1 |     0.1713 |     0.1217 | prob_std=0.0470
     2 |     0.1490 |     0.1103 | prob_std=0.0648
     3 |     0.1371 |     0.1028 | prob_std=0.0776
     4 |     0.1244 |     0.0971 | prob_std=0.0879
     5 |     0.1210 |     0.0918 | prob_std=0.0978
     6 |     0.1176 |     0.0877 | prob_std=0.1057
     7 |     0.1090 |     0.0835 | prob_std=0.1135
     8 |     0.1088 |     0.0803 | prob_std=0.1206
     9 |     0.1023 |     0.0762 | prob_std=0.1291
    10 |     0.1020 |     0.0732 | prob_std=0.1371
    11 |     0.0966 |     0.0703 | prob_std=0.1448
    12 |     0.0943 |     0.0671 | prob_std=0.1517
    13 |     0.0898 |     0.0640 | prob_std=0.1606


### Use the CNN to classify a catalog

In [17]:
import warnings
import numpy as np
import torch
import torch.nn as nn

from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from astropy.stats import mad_std

import astropy.units as u
import configparser


config = configparser.ConfigParser()
config.read("config.ini")

warnings.filterwarnings("ignore", category=UserWarning)

CUBE_FILE  = "./cube.fits"
MODEL_FILE = "lae_model.pt"

INPUT_CATALOG  = "./mawatari_matched.fits"
OUTPUT_CATALOG = "mawatari_mit_cnn_score.fits"

COL_RA       = "RA"
COL_DEC      = "DEC"
COL_REDSHIFT = "z"
COL_OUTPUT   = "CNN_Probability"

LYA_REST = 1215.67
HALF_Z = config.getint("TRAINING", "HALF_Z")
HALF_Y = config.getint("TRAINING", "HALF_Y")
HALF_X = config.getint("TRAINING", "HALF_X")
DROPOUT = config.getfloat("CNN", "DROPOUT")


def get_wcs_axis_order(wcs):

    axis_names = [n.upper() for n in wcs.axis_type_names]

    ra_ax = next(
        (i for i, n in enumerate(axis_names) if "RA" in n),
        None,
    )

    dec_ax = next(
        (i for i, n in enumerate(axis_names) if "DEC" in n),
        None,
    )

    wave_ax = next(
        (
            i
            for i, n in enumerate(axis_names)
            if any(k in n for k in ("WAVE", "LAMBDA"))
        ),
        None,
    )

    if None in (ra_ax, dec_ax, wave_ax):
        raise RuntimeError("WCS axes not found")

    return ra_ax, dec_ax, wave_ax


def normalize_subcube(sub):

    sub = np.nan_to_num(
        sub.astype(np.float32),
        nan=0.0,
    )

    local_std = mad_std(sub) + 1e-6

    sub = sub / local_std

    return sub


def score_with_tta(model, subcube, device):

    tensor = torch.tensor(
        subcube,
        dtype=torch.float32,
    ).unsqueeze(0).unsqueeze(0)

    scores = []

    with torch.no_grad():

        for k in range(4):

            t = torch.rot90(tensor, k=k, dims=[3, 4])

            score = torch.sigmoid(
                model(t.to(device))
            ).item()

            scores.append(score)

            t_flip = torch.flip(t, dims=[3])

            score_flip = torch.sigmoid(
                model(t_flip.to(device))
            ).item()

            scores.append(score_flip)

    return float(np.mean(scores))


def score_catalog(input_catalog, output_path):

    #device = torch.device(
    #    "cuda" if torch.cuda.is_available() else "cpu"
    #)
    device = torch.device("cpu")

    print(f"Device: {device}\n")

    print(f"Loading model: {MODEL_FILE}")

    model = LAEDetector3D().to(device)  # Has to be changed according to the training

    model.load_state_dict(
        torch.load(MODEL_FILE, map_location=device)
    )

    model.eval()

    print(f"Loading cube: {CUBE_FILE}")

    with fits.open(CUBE_FILE, memmap=False) as hdul:

        data_hdu = next(
            (h for h in hdul if h.data is not None and h.data.ndim == 3),
            hdul[0],
        )

        cube_data = np.array(data_hdu.data, dtype=np.float32)
        cube_data = np.nan_to_num(cube_data, nan=0.0)

        wcs = WCS(data_hdu.header)

    size_gb = cube_data.nbytes / 1e9

    print(f"Cube shape: {cube_data.shape}")
    print(f"RAM usage : {size_gb:.2f} GB")

    ra_ax, dec_ax, wave_ax = get_wcs_axis_order(wcs)

    max_z, max_y, max_x = cube_data.shape

    dz = HALF_Z
    dy = HALF_Y
    dx = HALF_X

    print(f"\nScoring catalog: {input_catalog}")

    cat = Table.read(input_catalog)

    n_sources = len(cat)

    probabilities = np.full(n_sources, np.nan, dtype=np.float32)

    skipped = 0

    # Debug: erste Quelle manuell prüfen
    row = cat[0]
    ra   = float(row[COL_RA])
    dec  = float(row[COL_DEC])
    z    = float(row[COL_REDSHIFT])
    
    wave_obs = LYA_REST * (1 + z)
    
    world = [None, None, None]
    world[ra_ax]   = ra
    world[dec_ax]  = dec
    world[wave_ax] = wave_obs
    
    pixel = wcs.all_world2pix([world], 0)[0]
    
    px = int(round(pixel[ra_ax]))
    py = int(round(pixel[dec_ax]))
    pz = int(round(pixel[wave_ax]))
    
    print(f"RA={ra:.4f}, DEC={dec:.4f}, z={z:.4f}")
    print(f"wave_obs={wave_obs:.2f}")
    print(f"world coords: {world}")
    print(f"pixel coords: px={px}, py={py}, pz={pz}")
    print(f"Cube shape (z,y,x): {max_z}, {max_y}, {max_x}")
    print(f"Bounds check:")
    print(f"  z: {dz} <= {pz} < {max_z - dz}  -> {dz <= pz < max_z - dz}")
    print(f"  y: {dy} <= {py} < {max_y - dy}  -> {dy <= py < max_y - dy}")
    print(f"  x: {dx} <= {px} < {max_x - dx}  -> {dx <= px < max_x - dx}")
    print(f"WCS axis order: ra_ax={ra_ax}, dec_ax={dec_ax}, wave_ax={wave_ax}")
    print(f"WCS axis names: {wcs.axis_type_names}")

    for i, row in enumerate(cat):

        if (i + 1) % max(1, n_sources // 20) == 0:
            print(
                f"Progress: {i+1}/{n_sources} "
                f"({100*(i+1)/n_sources:.0f}%)",
                end="\r",
            )

        try:

            ra   = float(row[COL_RA])
            dec  = float(row[COL_DEC])
            z    = float(row[COL_REDSHIFT])

            wave_obs = LYA_REST * (1 + z)

            world = [None, None, None]
            world[ra_ax]   = ra
            world[dec_ax]  = dec
            world[wave_ax] = wave_obs

            pixel = wcs.all_world2pix([world], 0)[0]

            px = int(round(pixel[ra_ax]))
            py = int(round(pixel[dec_ax]))
            pz = int(round(pixel[wave_ax]))

            if not (
                dz <= pz < max_z - dz and
                dy <= py < max_y - dy and
                dx <= px < max_x - dx
            ):
                skipped += 1
                continue

            sub = cube_data[
                pz-dz:pz+dz+1,
                py-dy:py+dy+1,
                px-dx:px+dx+1,
            ]

            sub = normalize_subcube(sub)

            score = score_with_tta(model, sub, device)

            probabilities[i] = score

        except Exception as e:
            print(f"Source {i} ({ra:.4f}, {dec:.4f}, z={z:.4f}): {type(e).__name__}: {e}")
            skipped += 1
        continue

    print(" " * 80, end="\r")

    if skipped > 0:
        print(f"Skipped sources: {skipped}")

    cat[COL_OUTPUT] = probabilities

    cat.sort(COL_OUTPUT)
    cat.reverse()

    cat.write(output_path, overwrite=True)

    evaluated = np.sum(~np.isnan(probabilities))

    print(f"\nEvaluated : {evaluated}/{n_sources}")
    print(f"Skipped   : {skipped}")
    print(f"Saved to  : {output_path}")


if __name__ == "__main__":

    score_catalog(
        input_catalog=INPUT_CATALOG,
        output_path=OUTPUT_CATALOG,
    )

Device: cpu

Loading model: lae_model.pt
Loading cube: ./cube.fits
Cube shape: (1036, 2478, 2260)
RAM usage : 23.21 GB

Scoring catalog: ./mawatari_matched.fits
RA=334.3852, DEC=0.1926, z=3.0690
wave_obs=4946.56
world coords: [334.385249, 0.192603, 4946.56123]
pixel coords: px=572, py=621, pz=738
Cube shape (z,y,x): 1036, 2478, 2260
Bounds check:
  z: 5 <= 738 < 1031  -> True
  y: 5 <= 621 < 2473  -> True
  x: 5 <= 572 < 2255  -> True
WCS axis order: ra_ax=0, dec_ax=1, wave_ax=2
WCS axis names: ['RA', 'DEC', 'Wave']
                                                                                
Evaluated : 168/168
Skipped   : 0
Saved to  : mawatari_mit_cnn_score.fits


In [18]:
print("done")

done


### Use the CNN to find sources in the cube

In [1]:
import warnings
import numpy as np
import torch
import torch.nn as nn
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from sklearn.cluster import DBSCAN
import configparser
from torch.utils.data import Dataset, DataLoader

config = configparser.ConfigParser()
config.read("config.ini")

warnings.filterwarnings("ignore", category=UserWarning)

# ── Pfade ─────────────────────────────────────────────────────────────────────
CUBE_FILE   = "./cube.fits"
MODEL_FILE  = "lae_model.pt"
OUTPUT_FILE = "kandidaten_cube_search.fits"
TRUE_CAT    = "./regions_tabelle.fits"

# ── Parameter ─────────────────────────────────────────────────────────────────
LYA_REST      = 1215.67
HALF_Z        = config.getint("TRAINING", "HALF_Z")
HALF_Y        = config.getint("TRAINING", "HALF_Y")
HALF_X        = config.getint("TRAINING", "HALF_X")
DROPOUT       = config.getfloat("CNN", "DROPOUT")
STRIDE        = 2        # geringer Stride für maximale Genauigkeit
THRESHOLD     = config.getfloat("CNN", "THRESHOLD")
DBSCAN_EPS    = 5.0
DBSCAN_MIN    = 4        # filtert Einzelpunkt-Rauschen heraus
MATCH_RADIUS  = 3.0
MATCH_DZ      = 0.01
TOTAL_SOURCES = None

# ── PyTorch Dataset für paralleles CPU-Slicing ────────────────────────────────
class SlidingWindowDataset(Dataset):
    def __init__(self, cube_data, coords, dz, dy, dx):
        self.cube_data = cube_data
        self.coords = coords
        self.dz, self.dy, self.dx = dz, dy, dx

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        z, y, x = self.coords[idx]
        dz, dy, dx = self.dz, self.dy, self.dx
        
        # CPU schneidet hier parallel über num_workers aus
        sub = self.cube_data[
            z - dz : z + dz + 1,
            y - dy : y + dy + 1,
            x - dx : x + dx + 1
        ].astype(np.float32)
        
        return torch.tensor(sub).unsqueeze(0), torch.tensor([z, y, x])


# ── WCS-Hilfsfunktion ─────────────────────────────────────────────────────────
def get_wcs_axis_order(wcs: WCS):
    axis_names = [name.upper() for name in wcs.axis_type_names]
    print(f"  WCS-Achsen erkannt: {axis_names}")

    ra_ax   = next((i for i, n in enumerate(axis_names) if "RA"   in n), None)
    dec_ax  = next((i for i, n in enumerate(axis_names) if "DEC"  in n), None)
    wave_ax = next(
        (i for i, n in enumerate(axis_names)
         if any(k in n for k in ("WAVE", "LAMBDA", "FREQ", "VELO"))),
        None,
    )

    if None in (ra_ax, dec_ax, wave_ax):
        raise ValueError(
            f"Konnte RA/DEC/Wellenlängen-Achse nicht identifizieren. "
            f"Gefundene Achsen: {axis_names}"
        )
    return ra_ax, dec_ax, wave_ax


# ── GT-Diagnose ───────────────────────────────────────────────────────────────
def diagnose_gt_scores(
    model:       nn.Module,
    cube_data:   np.ndarray,
    wcs:         WCS,
    device:      torch.device,
    true_cat:    str = TRUE_CAT,
):
    print("\n── GT-Diagnose: CNN-Score direkt an bekannten Quellen ──────────────")
    try:
        truth = Table.read(true_cat, format="fits")
    except Exception as e:
        print(f"  ✗ Konnte True-Katalog nicht laden: {e}")
        return

    ra_ax, dec_ax, wave_ax = get_wcs_axis_order(wcs)
    max_z, max_y, max_x = cube_data.shape
    dz, dy, dx = HALF_Z, HALF_Y, HALF_X

    scores = []
    for row in truth:
        ra, dec, redshift = float(row["RA"]), float(row["DEC"]), float(row["REDSHIFT"])
        wave_obs = LYA_REST * (1.0 + redshift)

        world = [None, None, None]
        world[ra_ax]   = ra
        world[dec_ax]  = dec
        world[wave_ax] = wave_obs

        try:
            pixel = wcs.all_world2pix([world], 0)[0]
        except Exception:
            continue

        px = int(round(float(pixel[ra_ax])))
        py = int(round(float(pixel[dec_ax])))
        pz = int(round(float(pixel[wave_ax])))

        if not (dz <= pz < max_z - dz and dy <= py < max_y - dy and dx <= px < max_x - dx):
            continue

        sub = cube_data[pz - dz : pz + dz + 1, py - dy : py + dy + 1, px - dx : px + dx + 1]

        with torch.no_grad():
            tensor = torch.tensor(sub, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            score  = torch.sigmoid(model(tensor)).item()

        scores.append(score)

    if scores:
        above = sum(s >= THRESHOLD for s in scores)
        print(f"\n  Zusammenfassung: {above}/{len(scores)} über Threshold ({THRESHOLD})")
        print(f"  Mittel={np.mean(scores):.3f}  Min={np.min(scores):.3f}  Max={np.max(scores):.3f}")
        print(f"  → Max. erreichbarer Recall beim Sliding Window: {above/len(scores):.1%}")
    print("─────────────────────────────────────────────────────────────────────\n")


# ── Paralleles Sliding Window via DataLoader ──────────────────────────────────
def sliding_window_inference_parallel(
    model:       nn.Module,
    cube_data:   np.ndarray,
    device:      torch.device,
    num_workers: int = 8
) -> tuple[list, list, list, list]:
    
    max_z, max_y, max_x = cube_data.shape
    dz, dy, dx = HALF_Z, HALF_Y, HALF_X

    z_range = range(dz, max_z - dz, STRIDE)
    y_range = range(dy, max_y - dy, STRIDE)
    x_range = range(dx, max_x - dx, STRIDE)

    coords = [(z, y, x) for z in z_range for y in y_range for x in x_range]
    print(f"  {len(coords):,} Positionen werden evaluiert …")

    # Dataset & DataLoader vorbereiten (batch_size nutzt 300GB RAM aus)
    dataset = SlidingWindowDataset(cube_data, coords, dz, dy, dx)
    dataloader = DataLoader(
        dataset, 
        batch_size=16384, 
        shuffle=False, 
        num_workers=num_workers, 
        pin_memory=True if device.type == "cuda" else False
    )

    found_z, found_y, found_x, probs_out = [], [], [], []
    model.eval()

    with torch.no_grad():
        for i, (subs_tensor, batch_coords_tensor) in enumerate(dataloader):
            subs_tensor = subs_tensor.to(device)
            probs = torch.sigmoid(model(subs_tensor)).cpu().numpy().flatten()
            batch_coords = batch_coords_tensor.numpy()

            for (z, y, x), prob in zip(batch_coords, probs):
                if prob > THRESHOLD:
                    found_z.append(int(z))
                    found_y.append(int(y))
                    found_x.append(int(x))
                    probs_out.append(float(prob))

            # Zuverlässiges Update alle 5 Batches im Jupyter Notebook
            if i % 5 == 0:
                evaluated = min((i + 1) * dataloader.batch_size, len(coords))
                pct = 100 * evaluated / len(coords)
                print(f"  … {pct:5.1f}%  |  {len(found_x)} Treffer bisher", end="\r", flush=True)

    print()
    return found_z, found_y, found_x, probs_out


# ── Non-Maximum Suppression via DBSCAN ───────────────────────────────────────
def apply_nms(
    found_z: list, found_y: list, found_x: list, probs: list
) -> tuple[list, list, list, list]:
    if not found_x:
        print("  Keine Detektionen über dem Schwellenwert.")
        return [], [], [], []

    coords = np.array([found_z, found_y, found_x], dtype=float).T
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN).fit_predict(coords)

    out_z, out_y, out_x, out_p = [], [], [], []

    n_noise = int((labels == -1).sum())
    if n_noise > 0:
        print(f"  DBSCAN: {n_noise} Einzelpunkt-Detektionen verworfen (Rauschen).")

    for cluster_id in set(labels):
        if cluster_id == -1:
            continue
        mask = labels == cluster_id
        idxs = np.where(mask)[0]
        best = idxs[np.argmax(np.array(probs)[idxs])]
        out_z.append(found_z[best])
        out_y.append(found_y[best])
        out_x.append(found_x[best])
        out_p.append(probs[best])

    print(f"  NMS: {len(found_x)} Rohdetektionen → {len(out_x)} einzigartige Kandidaten.")
    return out_z, out_y, out_x, out_p


# ── Qualitätsevaluation (Kompakt ohne Einzelzeilen) ───────────────────────────
def evaluate_candidates(
    candidates:    Table,
    true_cat_path: str,
    match_radius:  float = MATCH_RADIUS,
    match_dz:      float = MATCH_DZ,
    total_sources: int   = None,
) -> dict:
    print("\n── Qualitätsevaluation ─────────────────────────────────────────────")

    try:
        truth = Table.read(true_cat_path, format="fits")
    except Exception as e:
        print(f"  ✗ Konnte True-Katalog nicht laden: {e}")
        return {}

    n_truth = total_sources if total_sources is not None else len(truth)
    n_cands = len(candidates)

    if n_cands == 0:
        print("  Keine Kandidaten zum Evaluieren.")
        return {}

    cand_coords  = SkyCoord(ra=candidates["RA"]  * u.deg, dec=candidates["DEC"] * u.deg)
    truth_coords = SkyCoord(ra=truth["RA"]  * u.deg, dec=truth["DEC"] * u.deg)

    matched_truth = set()
    tp_indices    = []

    for i, (cand, cz) in enumerate(zip(cand_coords, candidates["REDSHIFT"])):
        sep   = cand.separation(truth_coords).arcsecond
        dz    = np.abs(truth["REDSHIFT"] - float(cz))
        match = np.where((sep <= match_radius) & (dz <= match_dz))[0]
        if len(match) > 0:
            best = match[np.argmin(sep[match])]
            matched_truth.add(int(best))
            tp_indices.append(i)

    tp = len(set(tp_indices))
    fp = n_cands - tp
    fn = n_truth - len(matched_truth)

    purity = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1     = (2 * purity * recall / (purity + recall) if (purity + recall) > 0 else 0.0)

    print(f"  Matching: ≤{match_radius}\" Winkelabstand, |Δz|≤{match_dz}")
    print(f"  Echte Quellen im Katalog : {n_truth}")
    print(f"  Gefundene Kandidaten     : {n_cands}")
    print()
    print(f"  True Positives   (TP)    : {tp}")
    print(f"  False Positives (FP)     : {fp}")
    print(f"  False Negatives (FN)     : {fn}")
    print()
    print(f"  Purity     (Precision)   : {purity:.3f}   ({tp}/{tp+fp})")
    print(f"  Completeness (Recall)    : {recall:.3f}   ({tp}/{tp+fn})")
    print(f"  F1-Score                 : {f1:.3f}")
    print("─────────────────────────────────────────────────────────────────")

    return {"tp": tp, "fp": fp, "fn": fn, "purity": purity, "recall": recall, "f1": f1}


# ── Haupt-Pipeline ────────────────────────────────────────────────────────────
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Gerät: {device}\n")

    print(f"[1/5] Lade Modell: {MODEL_FILE}")
    model = LAEDetector() # has to be changed according to the training
    
    # Optional Multi-GPU Support aktivieren falls 2 Karten im Cluster gemietet sind
    if torch.cuda.device_count() > 1:
        print(f"  --> Nutze {torch.cuda.device_count()} GPUs parallel via DataParallel!")
        model = nn.DataParallel(model)
        
    model = model.to(device)
    model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
    model.eval()
    print("  Modell geladen.")

    print(f"\n[2/5] Lade IFU-Würfel: {CUBE_FILE}")
    with fits.open(CUBE_FILE) as hdul:
        data_hdu = next(
            (h for h in hdul if h.data is not None and h.data.ndim == 3), hdul[0]
        )
        cube_data = np.nan_to_num(data_hdu.data.astype(np.float32))
        wcs = WCS(data_hdu.header)
    print(f"  Würfelgröße: {cube_data.shape}  (Z × Y × X)")

    print(f"\n[3/5] GT-Diagnose vor dem Sliding Window …")
    diagnose_gt_scores(model, cube_data, wcs, device)

    print(f"[4/5] Paralleles Sliding-Window (Stride={STRIDE}, Schwelle={THRESHOLD}) …")
    found_z, found_y, found_x, probs = sliding_window_inference_parallel(
        model, cube_data, device, num_workers=12
    )
    print(f"  {len(found_x)} Pixel-Treffer über Schwellenwert {THRESHOLD}.")

    print("\n[4b/5] Non-Maximum Suppression via DBSCAN …")
    found_z, found_y, found_x, probs = apply_nms(found_z, found_y, found_x, probs)

    if not found_x:
        print("✗ Keine Kandidaten nach NMS gefunden.")
        return

    # Pixel → Weltkoordinaten transformieren
    ra_ax, dec_ax, wave_ax = get_wcs_axis_order(wcs)
    res_ra, res_dec, res_redshift = [], [], []

    for x, y_pix, z in zip(found_x, found_y, found_z):
        pixel = [None, None, None]
        pixel[ra_ax]   = x
        pixel[dec_ax]  = y_pix
        pixel[wave_ax] = z
        world = wcs.all_pix2world([pixel], 0)[0]
        res_ra.append(float(world[ra_ax]))
        res_dec.append(float(world[dec_ax]))
        wave_obs = float(world[wave_ax])
        res_redshift.append(wave_obs / LYA_REST - 1.0)

    t = Table(
        [res_ra, res_dec, res_redshift, probs],
        names=("RA", "DEC", "REDSHIFT", "Probability"),
    )
    t.sort("Probability")
    t.reverse()
    t.write(OUTPUT_FILE, format="fits", overwrite=True)

    print(f"\n✓ {len(t)} Kandidaten gespeichert: {OUTPUT_FILE}")

    print("\n[5/5] Evaluiere Kandidaten gegen Ground-Truth …")
    evaluate_candidates(t, TRUE_CAT, MATCH_RADIUS, MATCH_DZ, TOTAL_SOURCES)

    print(f"\n✓ Inference komplett abgeschlossen.")


if __name__ == "__main__":
    main()

Gerät: cuda

[1/5] Lade Modell: lae_model.pt
  Modell geladen.

[2/5] Lade IFU-Würfel: ./cube.fits
  Würfelgröße: (1036, 2478, 2260)  (Z × Y × X)

[3/5] GT-Diagnose vor dem Sliding Window …

── GT-Diagnose: CNN-Score direkt an bekannten Quellen ──────────────
  WCS-Achsen erkannt: ['RA', 'DEC', 'WAVE']

  Zusammenfassung: 3466/4251 über Threshold (0.8)
  Mittel=0.883  Min=0.121  Max=0.993
  → Max. erreichbarer Recall beim Sliding Window: 81.5%
─────────────────────────────────────────────────────────────────────

[4/5] Paralleles Sliding-Window (Stride=2, Schwelle=0.8) …
  712,172,250 Positionen werden evaluiert …
  …  37.4%  |  1960402 Treffer bisher

KeyboardInterrupt: 

In [3]:
import warnings
import numpy as np
import torch
import torch.nn as nn
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
from sklearn.cluster import DBSCAN
import configparser

config = configparser.ConfigParser()
config.read("config.ini")
warnings.filterwarnings("ignore", category=UserWarning)

CUBE_FILE   = "./cube.fits"
MODEL_FILE  = "lae_model.pt"
OUTPUT_FILE = "kandidaten_cube_search.fits"
TRUE_CAT    = "./regions_tabelle.fits"
LYA_REST    = 1215.67

HALF_Z        = config.getint("TRAINING", "HALF_Z")
HALF_Y        = config.getint("TRAINING", "HALF_Y")
HALF_X        = config.getint("TRAINING", "HALF_X")
DROPOUT       = config.getfloat("CNN", "DROPOUT")
STRIDE        = 2
THRESHOLD     = config.getfloat("CNN", "THRESHOLD")
DBSCAN_EPS    = 5.0
DBSCAN_MIN    = 4
MATCH_RADIUS  = 3.0
MATCH_DZ      = 0.01
TOTAL_SOURCES = None

# Batch-Größe für GPU/CPU – größer = schneller, aber mehr VRAM
BATCH_SIZE = 512 if torch.cuda.is_available() else 128

def get_wcs_axes(wcs: WCS):
    names = [n.upper() for n in wcs.axis_type_names]
    ra  = next((i for i, n in enumerate(names) if "RA"  in n), None)
    dec = next((i for i, n in enumerate(names) if "DEC" in n), None)
    wav = next((i for i, n in enumerate(names)
                if any(k in n for k in ("WAVE", "LAMBDA", "FREQ", "VELO"))), None)
    if None in (ra, dec, wav):
        raise ValueError(f"WCS-Achsen nicht erkannt: {names}")
    return ra, dec, wav


def diagnose_gt_scores(model, cube_data, wcs, device):
    print("\n── GT-Diagnose ──────────────────────────────────────────────────────")
    try:
        truth = Table.read(TRUE_CAT, format="fits")
    except Exception as e:
        print(f"  Fehler: {e}"); return

    ra_ax, dec_ax, wave_ax = get_wcs_axes(wcs)
    max_z, max_y, max_x = cube_data.shape
    dz, dy, dx = HALF_Z, HALF_Y, HALF_X
    scores = []

    for row in truth:
        world = [None, None, None]
        world[ra_ax]   = float(row["RA"])
        world[dec_ax]  = float(row["DEC"])
        world[wave_ax] = LYA_REST * (1.0 + float(row["REDSHIFT"]))
        try:
            px = wcs.all_world2pix([world], 0)[0]
        except Exception:
            continue
        x, y, z = int(round(px[ra_ax])), int(round(px[dec_ax])), int(round(px[wave_ax]))
        if not (dz <= z < max_z-dz and dy <= y < max_y-dy and dx <= x < max_x-dx):
            continue
        sub = torch.tensor(
            cube_data[z-dz:z+dz+1, y-dy:y+dy+1, x-dx:x+dx+1], dtype=torch.float32
        ).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            scores.append(torch.sigmoid(model(sub)).item())

    if scores:
        above = sum(s >= THRESHOLD for s in scores)
        print(f"  {above}/{len(scores)} über Threshold | "
              f"Mean={np.mean(scores):.3f}  Min={np.min(scores):.3f}  Max={np.max(scores):.3f}")
        print(f"  Max. Recall beim Sliding Window: {above/len(scores):.1%}")
    print("─────────────────────────────────────────────────────────────────────\n")


def sliding_window_inference(model, cube_data, device):
    max_z, max_y, max_x = cube_data.shape
    dz, dy, dx = HALF_Z, HALF_Y, HALF_X

    # Stufe 1: Vorfilter – nur Positionen mit ausreichend hellem Signal
    print("  [1/2] Vorfilter …")
    bright = cube_data > 0.15
    zz, yy, xx = np.where(bright)
    stride_mask = (zz % STRIDE == 0) & (yy % STRIDE == 0) & (xx % STRIDE == 0)
    border_mask = (zz >= dz) & (zz < max_z-dz) & \
                  (yy >= dy) & (yy < max_y-dy) & \
                  (xx >= dx) & (xx < max_x-dx)
    mask = stride_mask & border_mask
    coords = np.stack([zz[mask], yy[mask], xx[mask]], axis=1)

    if len(coords) == 0:
        print("  Vorfilter zu streng – Fallback auf Vollgitter.")
        zr = np.arange(dz, max_z-dz, STRIDE*2)
        yr = np.arange(dy, max_y-dy, STRIDE*2)
        xr = np.arange(dx, max_x-dx, STRIDE*2)
        gz, gy, gx = np.meshgrid(zr, yr, xr, indexing="ij")
        coords = np.stack([gz.ravel(), gy.ravel(), gx.ravel()], axis=1)

    print(f"  [2/2] CNN auf {len(coords):,} Positionen (batch={BATCH_SIZE}) …")

    # Stufe 2: Batched Inference direkt ohne DataLoader-Overhead
    found_z, found_y, found_x, probs_out = [], [], [], []
    model.eval()
    n = len(coords)

    with torch.no_grad():
        for start in range(0, n, BATCH_SIZE):
            batch_coords = coords[start : start + BATCH_SIZE]
            # Subvolumen als vorallozierter Tensor
            B = len(batch_coords)
            patch_shape = (2*dz+1, 2*dy+1, 2*dx+1)
            batch_tensor = torch.empty((B, 1, *patch_shape), dtype=torch.float32, device=device)

            for i, (z, y, x) in enumerate(batch_coords):
                batch_tensor[i, 0] = torch.from_numpy(
                    cube_data[z-dz:z+dz+1, y-dy:y+dy+1, x-dx:x+dx+1]
                )

            probs = torch.sigmoid(model(batch_tensor)).cpu().numpy().flatten()
            hits  = probs > THRESHOLD

            found_z.extend(batch_coords[hits, 0].tolist())
            found_y.extend(batch_coords[hits, 1].tolist())
            found_x.extend(batch_coords[hits, 2].tolist())
            probs_out.extend(probs[hits].tolist())

            if (start // BATCH_SIZE) % 20 == 0:
                pct = 100 * min(start + BATCH_SIZE, n) / n
                print(f"  … {pct:5.1f}%  |  {len(found_x)} Treffer", end="\r", flush=True)

    print()
    return found_z, found_y, found_x, probs_out


def apply_nms(found_z, found_y, found_x, probs):
    if not found_x:
        print("  Keine Detektionen über Schwellenwert.")
        return [], [], [], []

    coords = np.array([found_z, found_y, found_x], dtype=float).T
    labels = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN).fit_predict(coords)

    out_z, out_y, out_x, out_p = [], [], [], []
    noise = int((labels == -1).sum())
    if noise:
        print(f"  DBSCAN: {noise} Rausch-Detektionen verworfen.")

    for cid in set(labels):
        if cid == -1:
            continue
        idxs = np.where(labels == cid)[0]
        best = idxs[np.argmax(np.array(probs)[idxs])]
        out_z.append(found_z[best]); out_y.append(found_y[best])
        out_x.append(found_x[best]); out_p.append(probs[best])

    print(f"  NMS: {len(found_x)} Rohdetektionen → {len(out_x)} Kandidaten.")
    return out_z, out_y, out_x, out_p


def evaluate_candidates(candidates, true_cat_path, total_sources=None):
    print("\n── Evaluation ──────────────────────────────────────────────────────")
    try:
        truth = Table.read(true_cat_path, format="fits")
    except Exception as e:
        print(f"  Fehler: {e}"); return {}

    n_truth = total_sources or len(truth)
    n_cands = len(candidates)
    if n_cands == 0:
        print("  Keine Kandidaten."); return {}

    cand_sc  = SkyCoord(ra=candidates["RA"]*u.deg, dec=candidates["DEC"]*u.deg)
    truth_sc = SkyCoord(ra=truth["RA"]*u.deg,      dec=truth["DEC"]*u.deg)

    matched_truth, tp_set = set(), set()
    for i, (c, cz) in enumerate(zip(cand_sc, candidates["REDSHIFT"])):
        sep   = c.separation(truth_sc).arcsecond
        dz    = np.abs(truth["REDSHIFT"] - float(cz))
        match = np.where((sep <= MATCH_RADIUS) & (dz <= MATCH_DZ))[0]
        if len(match):
            best = match[np.argmin(sep[match])]
            matched_truth.add(int(best)); tp_set.add(i)

    tp = len(tp_set); fp = n_cands - tp; fn = n_truth - len(matched_truth)
    purity = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1     = 2*purity*recall / (purity+recall) if purity+recall else 0.0

    print(f"  TP={tp}  FP={fp}  FN={fn}")
    print(f"  Purity={purity:.3f}  Recall={recall:.3f}  F1={f1:.3f}")
    print("─────────────────────────────────────────────────────────────────────")
    return {"tp": tp, "fp": fp, "fn": fn, "purity": purity, "recall": recall, "f1": f1}


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")

    print(f"[1/5] Lade Modell …")
    model = LAEDetector3D()
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)
    model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
    model.eval()

    print(f"[2/5] Lade Cube: {CUBE_FILE}")
    with fits.open(CUBE_FILE) as hdul:
        hdu = next((h for h in hdul if h.data is not None and h.data.ndim == 3), hdul[0])
        cube_data = np.nan_to_num(hdu.data.astype(np.float32))
        wcs = WCS(hdu.header)
    print(f"  Shape: {cube_data.shape}")

    print("\n[3/5] GT-Diagnose …")
    diagnose_gt_scores(model, cube_data, wcs, device)

    print("[4/5] Inferenz …")
    found_z, found_y, found_x, probs = sliding_window_inference(model, cube_data, device)
    print(f"  {len(found_x)} Treffer über Threshold {THRESHOLD}.")

    print("\n[4b/5] NMS …")
    found_z, found_y, found_x, probs = apply_nms(found_z, found_y, found_x, probs)

    if not found_x:
        print("Keine Kandidaten nach NMS."); return

    ra_ax, dec_ax, wave_ax = get_wcs_axes(wcs)
    res_ra, res_dec, res_z = [], [], []
    for x, y, z in zip(found_x, found_y, found_z):
        px = [None, None, None]
        px[ra_ax] = x; px[dec_ax] = y; px[wave_ax] = z
        world = wcs.all_pix2world([px], 0)[0]
        res_ra.append(float(world[ra_ax]))
        res_dec.append(float(world[dec_ax]))
        res_z.append(float(world[wave_ax]) / LYA_REST - 1.0)

    t = Table([res_ra, res_dec, res_z, probs], names=("RA", "DEC", "REDSHIFT", "Probability"))
    t.sort("Probability"); t.reverse()
    t.write(OUTPUT_FILE, format="fits", overwrite=True)
    print(f"\n✓ {len(t)} Kandidaten → {OUTPUT_FILE}")

    print("\n[5/5] Evaluation …")
    evaluate_candidates(t, TRUE_CAT, TOTAL_SOURCES)
    print("\n✓ Fertig.")


if __name__ == "__main__":
    main()

Device: cuda

[1/5] Lade Modell …
[2/5] Lade Cube: ./cube.fits
  Shape: (1036, 2478, 2260)

[3/5] GT-Diagnose …

── GT-Diagnose ──────────────────────────────────────────────────────
  2856/4251 über Threshold | Mean=0.818  Min=0.239  Max=0.951
  Max. Recall beim Sliding Window: 67.2%
─────────────────────────────────────────────────────────────────────

[4/5] Inferenz …
  [1/2] Vorfilter …


KeyboardInterrupt: 

In [ ]:
print("done")